In [1]:
import os
os.makedirs("delivery", exist_ok=True)
os.makedirs("data", exist_ok=True)
print("Folders ready")

Folders ready


In [2]:
%%writefile delivery/data.py
import pandas as pd
import numpy as np
import os

def load_data():
    path = "data/delivery_times.csv"

    if not os.path.exists(path):
        np.random.seed(42)
        n = 600
        distance_km = np.random.uniform(0.5, 12, n)
        prep_time_min = np.random.uniform(5, 30, n)
        traffic_level = np.random.randint(1, 4, n)
        rain = np.random.randint(0, 2, n)
        noise = np.random.normal(0, 2, n)

        delivery_min = 6 + 3 * distance_km + 0.6 * prep_time_min + 4 * traffic_level + 5 * rain + noise

        df = pd.DataFrame({
            "distance_km": distance_km,
            "prep_time_min": prep_time_min,
            "traffic_level": traffic_level,
            "rain": rain,
            "delivery_min": delivery_min
        })
        df.to_csv(path, index=False)

    return pd.read_csv(path)

Writing delivery/data.py


In [3]:
%%writefile delivery/features.py
def get_features_and_target(df):
    X = df[["distance_km", "prep_time_min", "traffic_level", "rain"]]
    y = df["delivery_min"]
    return X, y

# T1
def average_speed_kmph(distance_km, delivery_min):
    hours = delivery_min / 60
    return distance_km / hours

Writing delivery/features.py


In [4]:
%%writefile delivery/model.py
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
import joblib

def train_and_save_model(X, y):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    model = LinearRegression()
    model.fit(X_train, y_train)

    preds = model.predict(X_test)
    mae = mean_absolute_error(y_test, preds)
    print("Test MAE:", round(mae, 2))

    joblib.dump(model, "delivery_model.joblib")
    return model

Writing delivery/model.py


In [5]:
%%writefile delivery/validate.py
# T2
def is_valid_order(distance_km, prep_time_min, traffic_level, rain):
    if distance_km <= 0:
        return False
    if prep_time_min <= 0:
        return False
    if traffic_level not in [1, 2, 3]:
        return False
    if rain not in [0, 1]:
        return False
    return True

Writing delivery/validate.py


In [6]:
%%writefile delivery/__init__.py
from .data import load_data
from .features import get_features_and_target, average_speed_kmph
from .model import train_and_save_model
from .validate import is_valid_order

Writing delivery/__init__.py


In [7]:
from delivery import load_data, get_features_and_target, train_and_save_model

df = load_data()
X, y = get_features_and_target(df)
model = train_and_save_model(X, y)

Test MAE: 1.4


In [8]:
%%writefile train.py
from delivery import load_data, get_features_and_target, train_and_save_model

df = load_data()
X, y = get_features_and_target(df)
model = train_and_save_model(X, y)
print("Training done")

Writing train.py


In [9]:
!python train.py

Test MAE: 1.4
Training done


In [10]:
%%writefile predict.py
import sys
import joblib
import pandas as pd
from delivery import is_valid_order

distance_km = float(sys.argv[1])
prep_time_min = float(sys.argv[2])
traffic_level = int(sys.argv[3])
rain = int(sys.argv[4])

if not is_valid_order(distance_km, prep_time_min, traffic_level, rain):
    print("Invalid order")
else:
    model = joblib.load("delivery_model.joblib")
    order = pd.DataFrame([[distance_km, prep_time_min, traffic_level, rain]],
                          columns=["distance_km", "prep_time_min", "traffic_level", "rain"])
    prediction = model.predict(order)[0]
    print("Predicted delivery time:", round(prediction, 1), "minutes")

Writing predict.py


In [11]:
!python predict.py 5.0 15 2 0

Predicted delivery time: 37.9 minutes


In [12]:
!python predict.py -3 15 2 0

Invalid order


In [13]:
from delivery import average_speed_kmph

speed = average_speed_kmph(distance_km=5.0, delivery_min=40.4)
print("Average speed (km/h):", round(speed, 2))

Average speed (km/h): 7.43


In [14]:
import os
os.makedirs("cancer", exist_ok=True)
print("cancer package folder ready")

cancer package folder ready


In [15]:
%%writefile cancer/data.py
from sklearn.datasets import load_breast_cancer
import pandas as pd

def load_data():
    bc = load_breast_cancer()
    df = pd.DataFrame(bc.data, columns=bc.feature_names)
    df["target"] = bc.target        # 0 = malignant, 1 = benign
    return df

Writing cancer/data.py


In [16]:
%%writefile cancer/features.py
def get_features_and_target(df):
    X = df.drop(columns=["target"])
    y = df["target"]
    return X, y

Writing cancer/features.py


In [17]:
%%writefile cancer/model.py
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score
import joblib

# T1: compare models with cross-validation, save whichever scores best
def train_and_save_model(X, y):
    candidates = {
        "LogisticRegression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000)),
        "DecisionTree": DecisionTreeClassifier(random_state=42),
        "RandomForest": RandomForestClassifier(n_estimators=100, random_state=42),
    }

    scores = {}
    for name, model in candidates.items():
        cv = cross_val_score(model, X, y, cv=5, scoring="accuracy")
        scores[name] = cv.mean()
        print(name, "CV accuracy", round(cv.mean(), 3))

    best_name = max(scores, key=scores.get)
    print("Best model:", best_name)

    best_model = candidates[best_name]
    best_model.fit(X, y)
    joblib.dump(best_model, "cancer_model.joblib")
    return best_model

Writing cancer/model.py


In [18]:
%%writefile cancer/validate.py
# T2: reject unless at least 3 measurements are in a realistic range
def is_valid_order(radius_mean, texture_mean, perimeter_mean, area_mean):
    checks = [
        0 < radius_mean < 50,
        0 < texture_mean < 60,
        0 < perimeter_mean < 300,
        0 < area_mean < 5000,
    ]
    return sum(checks) >= 3

Writing cancer/validate.py


In [19]:
%%writefile cancer/metrics.py
# T4: report confusion matrix + classification report for the saved model
import joblib
from sklearn.metrics import confusion_matrix, classification_report

def report_performance(X, y):
    model = joblib.load("cancer_model.joblib")
    preds = model.predict(X)
    print("Confusion matrix:")
    print(confusion_matrix(y, preds))
    print()
    print("Classification report:")
    print(classification_report(y, preds, target_names=["malignant", "benign"]))

Writing cancer/metrics.py


In [20]:
%%writefile cancer/__init__.py
from .data import load_data
from .features import get_features_and_target
from .model import train_and_save_model
from .validate import is_valid_order
from .metrics import report_performance

Writing cancer/__init__.py


In [21]:
from cancer import load_data, get_features_and_target, train_and_save_model

df = load_data()
X, y = get_features_and_target(df)
model = train_and_save_model(X, y)

LogisticRegression CV accuracy 0.981
DecisionTree CV accuracy 0.917
RandomForest CV accuracy 0.956
Best model: LogisticRegression


In [22]:
%%writefile train_cancer.py
from cancer import load_data, get_features_and_target, train_and_save_model

df = load_data()
X, y = get_features_and_target(df)
model = train_and_save_model(X, y)
print("Training done")

Writing train_cancer.py


In [23]:
!python train_cancer.py

LogisticRegression CV accuracy 0.981
DecisionTree CV accuracy 0.917
RandomForest CV accuracy 0.956
Best model: LogisticRegression
Training done


In [24]:
%%writefile predict_cancer.py
import sys
import joblib
import pandas as pd
from sklearn.datasets import load_breast_cancer
from cancer import is_valid_order

# 4 key measurements from the command line
radius_mean = float(sys.argv[1])
texture_mean = float(sys.argv[2])
perimeter_mean = float(sys.argv[3])
area_mean = float(sys.argv[4])

if not is_valid_order(radius_mean, texture_mean, perimeter_mean, area_mean):
    print("Invalid order")
    sys.exit()

# build a full 30-feature row: fill the rest with dataset means, override the 4 given
bc = load_breast_cancer()
row = pd.DataFrame(bc.data, columns=bc.feature_names).mean()
row["mean radius"] = radius_mean
row["mean texture"] = texture_mean
row["mean perimeter"] = perimeter_mean
row["mean area"] = area_mean
order = pd.DataFrame([row])

model = joblib.load("cancer_model.joblib")
pred = int(model.predict(order)[0])
proba = model.predict_proba(order)[0]

label = "benign" if pred == 1 else "malignant"
confidence = proba[pred] * 100
# T3: print prediction AND confidence
print("Prediction:", label)
print("Confidence:", round(confidence, 1), "%")

Writing predict_cancer.py


In [25]:
!python predict_cancer.py 14 20 90 600

Prediction: benign
Confidence: 56.7 %


In [26]:
!python predict_cancer.py 999 999 90 600

Invalid order


In [27]:
from cancer import load_data, get_features_and_target, report_performance

df = load_data()
X, y = get_features_and_target(df)
report_performance(X, y)

Confusion matrix:
[[207   5]
 [  2 355]]

Classification report:
              precision    recall  f1-score   support

   malignant       0.99      0.98      0.98       212
      benign       0.99      0.99      0.99       357

    accuracy                           0.99       569
   macro avg       0.99      0.99      0.99       569
weighted avg       0.99      0.99      0.99       569

